# Fine-tuning Qwen3-Embedding-4B with LoRA

**Question (Further Work):** the frozen Qwen3-Embedding-4B is the best model so far: 78.97 on test with a tuned MLP on its vectors. Fine-tuning all of RoBERTa's weights lifted it from 68.33 (frozen + MLP) to about 72 (`Fine-tuned Roberta.ipynb`, `fine-tuning-results.md`). Does Qwen gain too when it adapts to the task?

Training all of Qwen's weights, the way we trained RoBERTa's, doesn't fit on this GPU. **LoRA** (low-rank adaptation) gets around that. Qwen's 4 billion weights stay frozen, and we train small extra matrices added to its layers: 33M numbers in all, 0.8% of the model.

Same data, test set and thread-grouped dev split as the RoBERTa notebook. Qwen reads each post as in the frozen notebooks: the best instruction in front, and the last token's vector as the post's summary.

**Before running:** it needs about 10 GB of GPU memory (9.6 GB measured), so don't run it alongside another GPU job (such as a fine-tuning sweep). It took about 53 minutes on the 5070 Ti: 16.4 minutes per epoch, plus the test set.

**Predict before running:** where will LoRA-fine-tuned Qwen3-Embedding-4B land?
- Frozen Qwen3-Embedding-4B + tuned MLP: 78.97
- Sada et al.'s LoRA fine-tune of the general Qwen3-8B chat model, on our test posts: 77.75
- RoBERTa-base, fine-tuned: about 72

## Why not fine-tune all of Qwen, as with RoBERTa?

Memory. For each weight it trains, AdamW keeps the weight itself (32-bit: 4 bytes), its gradient (4 bytes) and two running averages (8 bytes): 16 bytes per weight.

| | weights | memory to train them all |
|---|---|---|
| RoBERTa-base | 125M | 2 GB |
| Qwen3-Embedding-4B | 4,022M | 64 GB |

The GPU has 16 GB, and the other programs on this machine use 3–5 GB of it. Just storing Qwen's weights in bfloat16 takes 8 GB.

## LoRA

**Read first:** J&M Section 8.2, "Parameter Efficient Fine Tuning" (pp. 213–215, in the Post-training chapter, with Figures 8.7 and 8.8). It's about two pages. The lectures don't cover LoRA.

The idea, in the notation used below:
- A transformer's big weights are matrices that multiply vectors: $y = Wx$. Each of Qwen's 36 layers has 7 of them: q, k, v and o in self-attention, and gate, up and down in the feed-forward part.
- Fine-tuning changes each $W$ to $W + \Delta W$, and $\Delta W$ is as big as $W$.
- **LoRA's bet:** the change one task needs has low rank, so it can be written as the product of two thin matrices, $\Delta W = BA$:
  - $A$ is $r \times d_\text{in}$: it maps the input ($d_\text{in}$ numbers) down to $r$ numbers.
  - $B$ is $d_\text{out} \times r$: it maps those $r$ numbers up to the output.
  - We use $r = 16$, while $d_\text{in}$ and $d_\text{out}$ are 1,024 to 9,728.
- $W$ stays frozen. Only $A$ and $B$ train. The layer computes

$$y = Wx + \frac{\alpha}{r} BAx$$

**Example:** layer 1's query projection. $W$ is 4,096 × 2,560, about 10.5M numbers. $A$ (16 × 2,560) and $B$ (4,096 × 16) together have 106,496, which is 1% of $W$.

**Two details J&M leaves out,** from the LoRA paper's Section 4.1:
- **$B$ starts at zero,** so $BA = 0$. Before the first step, the model computes exactly what pre-trained Qwen computes. $A$ starts random.
- **The update is scaled by $\alpha / r$** (here 32 / 16 = 2). Dividing by $r$ means you can try another $r$ without retuning the learning rate. $\alpha$ acts roughly as a multiplier on the learning rate, so it's usually set once and not tuned; $\alpha = 2r$ is a common rule of thumb.

J&M writes the same thing with row vectors, $h = xW + xAB$, so its $A$ and $B$ are the transposes of ours. In both, $A$ is the half that reads the input and $B$ the half that writes the output.

**What LoRA saves, and what it doesn't:**
- There are no gradients and no AdamW state for the 4 billion frozen weights. The trainable part (33M numbers) needs about 0.5 GB instead of 64 GB.
- The backward pass is cheaper too: it skips the frozen weights' gradients, so it costs about one forward pass instead of two.
- The frozen weights can be stored in bfloat16 (8 GB). The RoBERTa notebook kept its weights in 32-bit because a small update rounds away in bfloat16, but these weights are never updated.
- Backpropagation still runs through all 36 layers: layer 1's $A$ and $B$ need the gradient that starts at the loss on top. So the forward pass still has to keep every layer's intermediate results. That's the next memory problem (gradient checkpointing, below).

**Which matrices?** The LoRA paper adapted only $W_q$ and $W_v$ (Section 7.1), and Sada et al. adapted only the attention matrices. QLoRA (Dettmers et al. 2023, Section 4) later found that "LoRA on all linear transformer block layers are required to match full finetuning performance", so we adapt all 7.

**Further reading (optional):**
- The LoRA paper: Hu et al., ICLR 2022, https://arxiv.org/abs/2106.09685. Figure 1 and Section 4.1 (one page) are the method. Section 7 asks why such a small rank is enough.
- Sebastian Raschka, "Practical Tips for Finetuning LLMs Using LoRA": https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms. Experiments on $r$, $\alpha$ and which layers to adapt.
- LoRA is usually applied with Hugging Face's `peft` library (not installed here): https://huggingface.co/docs/peft/main/en/conceptual_guides/lora. Below we write it ourselves, in about 15 lines, as the RoBERTa notebook wrote out its classification head.

## Compared with fine-tuning RoBERTa

1. **The model:** Qwen3-Embedding-4B (36 layers, 2,560-d vectors) instead of RoBERTa-base (12 layers, 768-d), with a linear head on the last token's vector.
2. **LoRA:** all of Qwen's weights are frozen, and $A$ and $B$ are added to all 7 matrices of all 36 layers. The optimizer gets only $A$, $B$ and the head.
3. **Learning rate 1e-4 instead of 2e-5.** $A$ and $B$ have to grow from nothing, rather than nudge weights that are already good. The LoRA paper used 2e-4 to 5e-4. 1e-4 is at the careful end, because the new head learns at the same rate.
4. **Memory:** frozen weights in bfloat16, gradient checkpointing, and gradient accumulation (each explained where it appears).
5. **Speed:** batches of posts of similar length, so little compute goes on padding.

**The rest is the BERT paper's short recipe:** 3 epochs, batches of 16, AdamW with weight decay 0.01, warmup over 10% of the steps then linear decay, gradient clipping, and the weights after the last epoch. (The first RoBERTa runs kept the epoch with the lowest dev loss instead, which the RoBERTa sweep found picks too early; see the training loop.) Longer recipes, such as the RoBERTa paper's 10 epochs, would take over 2 hours with a 4B model.

## Load dataset

Same as the RoBERTa notebook, whose comments explain the Python.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'))

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'))

# Remove empty documents
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)      ## (11096,) int64: the class id (0-19) of each training post
y_test = np.array(newsgroups_test.target)        ## (7370,)
n_classes = len(newsgroups_train.target_names)   ## 20

Removed 218 empty documents from the train set. Before: 11314. After: 11096.
Removed 162 empty documents from the test set. Before: 7532. After: 7370.


## Dev set: 10% of the threads

The same split as the RoBERTa notebook (same seed, so the same 1,109 dev posts), so the dev scores can be compared. Replies share their thread's subject line, so a random split would put most dev posts' thread-mates in the training part. The headers are only read to find each post's thread; the model never sees them.

In [2]:
import re
from sklearn.model_selection import StratifiedGroupKFold

## Each post's thread: its subject line (from the headers) without any leading "Re:", lowercased
with_headers = fetch_20newsgroups(subset='train')
subject_of = {f: re.search(r"^Subject:(.*)$", doc, re.M).group(1)
              for f, doc in zip(with_headers.filenames, with_headers.data)}
threads = [re.sub(r"^(re:\s*)+", "", subject_of[f].strip(), flags=re.IGNORECASE).lower()
           for f in newsgroups_train.filenames]
thread_ids = np.unique(threads, return_inverse=True)[1]   ## (11096,): one integer per thread

## The first of 10 folds that never split a thread: 90% of the threads to train on, 10% for dev
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=423)
train_idx, dev_idx = next(splitter.split(newsgroups_train.data, y_train, groups=thread_ids))
X_tr, y_tr = [newsgroups_train.data[i] for i in train_idx], y_train[train_idx]   ## 9,987 strings, (9987,)
X_dev, y_dev = [newsgroups_train.data[i] for i in dev_idx], y_train[dev_idx]     ## 1,109 strings, (1109,)
print(f"Training part: {len(X_tr):,} posts, dev: {len(X_dev):,} posts, "
      f"threads in both: {len(set(thread_ids[train_idx]) & set(thread_ids[dev_idx]))}")

Training part: 9,987 posts, dev: 1,109 posts, threads in both: 0


## Tokenizing

- **The instruction goes in front of every post:** `Instruct: Classify the given Usenet newsgroup post by the newsgroup it was posted to\nQuery:`. Qwen3-Embedding was trained with an instruction in front of each query, and with the frozen model this one beat Qwen's own prompt for this dataset by about 1 point (`hparam_search_mean/Qwen4B2048Search.md`). It's 25 tokens.
- **The tokenizer puts `<|endoftext|>` at the end of every post.** That's the token whose vector the head reads (see the model).
- **Posts are cut at 512 tokens,** instruction and `<|endoftext|>` included. For the frozen model, reading 2,048 tokens instead of 512 gained only about 0.1 points (grouped CV), because most errors are on short posts. Every token costs training time. 10.5% of the training posts get cut.
- **Every post is tokenized once, up front,** because the batching below needs each post's length before it forms the batches.

In [3]:
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen3-Embedding-4B"
INSTRUCTION = "Instruct: Classify the given Usenet newsgroup post by the newsgroup it was posted to\nQuery:"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL)   ## about 151k subword tokens: byte-level BPE, as in RoBERTa

## A list of N strings -> a list of N lists of token ids, of different lengths (no padding yet).
## truncation=True with max_length cuts the end off a long post, keeping room for <|endoftext|>.
def tokenize(texts):
    return tokenizer([INSTRUCTION + t for t in texts], truncation=True, max_length=MAX_LENGTH)["input_ids"]

ids_tr, ids_dev, ids_test = tokenize(X_tr), tokenize(X_dev), tokenize(newsgroups_test.data)
lengths_tr = np.array([len(ids) for ids in ids_tr])   ## (9987,): each training post's length in tokens

print(f"Training posts: mean {lengths_tr.mean():.0f} tokens, median {np.median(lengths_tr):.0f}, "
      f"{(lengths_tr == MAX_LENGTH).mean() * 100:.1f}% cut at {MAX_LENGTH}. {lengths_tr.sum():,} tokens per epoch.")
## The first post's first 28 tokens (the instruction is 25) and its last 3. Ġ marks a space before a token and
## Ċ a newline, as in RoBERTa.
print(tokenizer.convert_ids_to_tokens(ids_tr[0][:28]), "...", tokenizer.convert_ids_to_tokens(ids_tr[0][-3:]))

/home/tonyxdsu/venvs/cs423/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training posts: mean 203 tokens, median 152, 10.5% cut at 512. 2,025,383 tokens per epoch.
['In', 'struct', ':', 'ĠClass', 'ify', 'Ġthe', 'Ġgiven', 'ĠUs', 'enet', 'Ġnew', 'sg', 'roup', 'Ġpost', 'Ġby', 'Ġthe', 'Ġnew', 'sg', 'roup', 'Ġit', 'Ġwas', 'Ġposted', 'Ġto', 'Ċ', 'Query', ':I', 'Ġwas', 'Ġwondering', 'Ġif'] ... ['-mail', '.', '<|endoftext|>']


## Batches of similar length, split into pieces

**Grouping by length.** A batch is padded up to its longest post, so one 512-token post makes 15 short ones pad to 512 too. With random batches of 16, 2.4× as many tokens would go through the model as the posts contain. Instead, each epoch (as `sweep.py` does, following fairseq, RoBERTa's own training code, and Hugging Face's `group_by_length`):
1. Shuffle the training posts and cut them into chunks of 800 (50 batches).
2. Sort each chunk by length and cut it into batches of 16. Posts in a batch now have similar lengths, and padding adds only 1%.
3. Shuffle the order of the batches, so long and short batches are mixed through the epoch.

Sorting within random chunks, rather than sorting the whole set once, gives different batches every epoch.

**Gradient accumulation.** A batch of 16 posts of 512 tokens is 8,192 tokens, too many to backpropagate at once in the memory we have (see the model). So a batch goes through in pieces of at most 2,048 tokens, padding included: 1 piece for a batch of short posts, up to 4 for the longest. Each piece's loss gets its own `backward()`. Since `backward()` adds into each weight's `.grad` (+=, the reason `zero_grad()` exists), after the last piece `.grad` holds the gradient for the whole batch: the same as if all 16 posts had gone through together, up to rounding. Then one optimizer step.

**Padding on the right.** Each row holds a post's tokens first and its padding after, with the attention mask (1 = real token, 0 = padding) telling Qwen to ignore the padding. The frozen notebooks padded on the left, so that every post's last token sat at position −1. Here the model finds each post's last token from the attention mask instead.

**Shapes.** From here on, the comments give the shape of every tensor:
- **N** = posts in a piece (B is one of LoRA's matrices in this notebook, so it isn't used for the batch size)
- **T** = tokens in the piece's longest post, at most 512
- **2560** = the size of Qwen's vectors; **20** = classes

In [4]:
device = "cuda"
BATCH_SIZE = 16          ## posts per optimizer step, as in the RoBERTa notebook
TOKENS_PER_PIECE = 2048  ## at most this many tokens per forward/backward pass, padding included. If the GPU runs out of
                         ## memory, halve it: batches just get split into more pieces, so the results stay the same.
rng = np.random.default_rng(423)   ## numpy's random number generator, seeded so that every run makes the same batches

## One epoch's batches: a list of 625 arrays, each holding the positions in ids_tr of up to 16 posts of similar length
def train_batches():
    order = rng.permutation(len(ids_tr))                    ## (9987,): 0, 1, ..., 9986 in random order
    batches = []
    for start in range(0, len(order), 50 * BATCH_SIZE):     ## start = 0, 800, 1600, ...
        chunk = order[start:start + 50 * BATCH_SIZE]         ## 800 random posts (the last chunk has fewer)
        ## np.argsort(v) returns the positions that would sort v, shortest first (like sorting an array of indices
        ## with a comparator in C++); chunk[...] then reorders the chunk by them.
        chunk = chunk[np.argsort(lengths_tr[chunk])]
        batches += [chunk[i:i + BATCH_SIZE] for i in range(0, len(chunk), BATCH_SIZE)]   ## += appends 50 batches
    return [batches[i] for i in rng.permutation(len(batches))]   ## the same batches, in a random order

## Splits one batch into pieces of at most TOKENS_PER_PIECE tokens. lengths: the posts' lengths, so that lengths[batch]
## are this batch's (lengths_tr for training; evaluation passes its own). No piece is padded beyond the batch's longest
## post, so a piece can hold TOKENS_PER_PIECE // (that length) posts: 2048 // 512 = 4 when the posts are the longest.
def pieces(batch, lengths):
    per_piece = max(1, TOKENS_PER_PIECE // lengths[batch].max())
    return [batch[i:i + per_piece] for i in range(0, len(batch), per_piece)]

## A list of N token-id lists -> input_ids and attention_mask, both (N, T), padded on the right, in GPU memory
def pad(seqs):
    T = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), T), tokenizer.pad_token_id)   ## (N, T), every entry the padding token's id
    attention_mask = torch.zeros((len(seqs), T), dtype=torch.long)   ## (N, T), all 0
    for i, s in enumerate(seqs):
        input_ids[i, :len(s)] = torch.tensor(s)   ## row i, columns 0 to len(s)-1: the post's tokens
        attention_mask[i, :len(s)] = 1
    return input_ids.to(device), attention_mask.to(device)

batches = train_batches()
print(f"{len(batches)} batches per epoch, {sum(len(pieces(b, lengths_tr)) for b in batches)} pieces")
## Three batches: the one with the longest posts, the first one, and the one with the shortest posts.
## max(items, key=f) returns the item with the largest f(item), min the smallest. "lambda b: ..." is an unnamed
## function, like the C++ lambda [](auto b) { return ...; }.
longest = max(batches, key=lambda b: lengths_tr[b].max())
shortest = min(batches, key=lambda b: lengths_tr[b].max())
for b in (longest, batches[0], shortest):
    print("post lengths", lengths_tr[b].tolist(), "-> pieces of", [len(p) for p in pieces(b, lengths_tr)], "posts")

input_ids, attention_mask = pad([ids_tr[i] for i in shortest])
print("the shortest batch, padded: input_ids", tuple(input_ids.shape), " attention_mask", tuple(attention_mask.shape))
## Where the post in row 0 (the batch's shortest) ends. The padding token is <|endoftext|> too, so only the attention
## mask tells the post's real last token from the padding after it.
n0 = int(attention_mask[0].sum())   ## row 0's number of real tokens: the 1s in its mask
print(f"row 0, positions {n0 - 2} to {n0 + 2}:", tokenizer.convert_ids_to_tokens(input_ids[0, n0 - 2:n0 + 3].tolist()),
      " mask:", attention_mask[0, n0 - 2:n0 + 3].tolist())

625 batches per epoch, 1291 pieces
post lengths [512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512] -> pieces of [4, 4, 4, 4] posts
post lengths [240, 242, 243, 247, 249, 249, 250, 251, 251, 253, 256, 258, 258, 260, 263, 268] -> pieces of [7, 7, 2] posts
post lengths [26, 26, 26, 26, 26, 26, 27, 27, 27, 29, 30, 31, 31, 31, 31, 32] -> pieces of [16] posts
the shortest batch, padded: input_ids (16, 32)  attention_mask (16, 32)


row 0, positions 24 to 28: [':ĊĊ', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>']  mask: [1, 1, 0, 0, 0]


## The model: Qwen3-Embedding-4B + LoRA + a linear head on the last token

**Which vector.** Qwen is a decoder-only transformer: its self-attention is causal, so each token attends only to itself and the tokens before it (Lecture 10). Only the last token, `<|endoftext|>`, has seen the whole post, and Qwen3-Embedding was trained to put a summary of the text into that token's vector. With right padding, a post's last real token is at position (number of real tokens − 1), which the attention mask gives: the number of 1s in its row, minus 1.

**The head:** dropout 0.1, then Linear(2560, 20), new and randomly initialized: the BERT paper's head, which the first RoBERTa runs used too.

**LoRA:** freeze all of Qwen's weights, then replace each of the 7 matrices in each of the 36 layers (252 in all) by a `LoRALinear` that keeps the original layer and adds $\frac{\alpha}{r} BAx$ to its output. Settings: $r = 16$, $\alpha = 32$, and dropout 0.05 on LoRA's input, on the way into $A$ only (QLoRA found that 0.05 helped for 7B and 13B models).

**Two precisions.** The frozen weights are bfloat16 (8 GB). The trainable ones ($A$, $B$, the head) are 32-bit, for the RoBERTa notebook's reason: one AdamW step is too small to show up in bfloat16. Inside `torch.autocast`, each matrix multiply converts both of its sides to bfloat16, so the two kinds can meet.

**Gradient checkpointing.** Backpropagation needs each layer's intermediate results from the forward pass. One Qwen layer with LoRA keeps about 220 KB per token (measured on the CPU with random weights), so 36 layers and a 2,048-token piece would need 16 GB. With gradient checkpointing, the forward pass keeps only each layer's input (5 KB per token) and throws the rest away. The backward pass then reruns each layer's forward pass from that input, just before backpropagating through the layer. Memory: 36 × 5 KB, plus 220 KB for the one layer being worked on, per token: about 0.8 GB for a 2,048-token piece. Cost: every layer's forward pass runs twice. With the frozen weights needing no gradients, a backward pass costs about as much as a forward pass, so a step does 3 passes' work instead of 2: about 50% more compute.

**Expected peak GPU memory:** 8 GB of frozen weights + 0.5 GB for the trainable weights, their gradients and AdamW's averages + about 1 GB of activations: about 10 GB. Measured: 9.6 GB.

In [5]:
import math
from transformers import AutoModel

R, ALPHA, LORA_DROPOUT = 16, 32, 0.05

## One pre-trained nn.Linear (frozen), plus LoRA's two small trainable matrices
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):   ## base: the pre-trained layer, whose weight W is (d_out, d_in)
        super().__init__()
        self.base = base
        ## nn.Linear(n_in, n_out) stores its weight as (n_out, n_in), like W, and maps each input vector x to weight @ x.
        ## device=...: create the weights directly in GPU memory, next to W. They're 32-bit, PyTorch's default.
        self.A = nn.Linear(base.in_features, r, bias=False, device=base.weight.device)   ## A: (r, d_in), random
        self.B = nn.Linear(r, base.out_features, bias=False, device=base.weight.device)  ## B: (d_out, r)
        nn.init.zeros_(self.B.weight)    ## B = 0, so B A = 0: the layer starts out computing exactly W x
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / r           ## 32 / 16 = 2

    def forward(self, x):                ## x: (N, T, d_in)
        ##     W x, as before  +  dropout -> A: (N, T, r) -> B: (N, T, d_out) -> times alpha / r
        return self.base(x) + self.B(self.A(self.dropout(x))) * self.scale   ## (N, T, d_out)

torch.manual_seed(423)   ## the same random A's, head and dropout masks in every run

## bfloat16 is the precision Qwen was trained in, and half the memory of 32-bit: 8 GB
qwen = AutoModel.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to(device)
qwen.requires_grad_(False)   ## freeze all of Qwen's weights: autograd won't compute gradients for them

## Replace each of the 7 matrices in each layer by a LoRALinear around it. getattr(obj, "q_proj") reads the attribute
## named by the string, obj.q_proj; setattr(obj, "q_proj", value) assigns obj.q_proj = value.
for layer in qwen.layers:            ## the 36 decoder layers
    for module, names in [(layer.self_attn, ["q_proj", "k_proj", "v_proj", "o_proj"]),
                          (layer.mlp, ["gate_proj", "up_proj", "down_proj"])]:
        for name in names:
            setattr(module, name, LoRALinear(getattr(module, name), R, ALPHA, LORA_DROPOUT))
## The A's and B's were created after requires_grad_(False), so they're trainable (new weights get requires_grad=True).

## Gradient checkpointing in each of the 36 layers; it's only active in model.train() mode. use_reentrant=False picks
## PyTorch's newer implementation. With the older one (Hugging Face's default), a checkpointed layer whose input doesn't
## require a gradient passes no gradient back. Here layer 1's input comes from the frozen embedding table, so the A's
## and B's would silently never train.
qwen.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})


class QwenClassifier(nn.Module):
    def __init__(self, qwen, n_classes):
        super().__init__()
        self.qwen = qwen
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(qwen.config.hidden_size, n_classes)   ## new and random: weight (20, 2560), bias (20,)

    def forward(self, input_ids, attention_mask):   ## both (N, T)
        ## Inside self.qwen, to draw it out:
        ##   input_ids (N, T) -> rows of a (151665, 2560) embedding table -> (N, T, 2560)
        ##   then 36 layers, each (N, T, 2560) -> (N, T, 2560):
        ##     self-attention: RMSNorm; q_proj -> (N, T, 4096) = 32 heads of 128; k_proj, v_proj -> (N, T, 1024) =
        ##                     8 heads of 128, each shared by 4 query heads; q and k get an RMSNorm per head and are
        ##                     rotated by each token's position (RoPE); scores = q kᵀ / √128, where each token sees
        ##                     only itself and earlier tokens, and no padding; softmax; @ v; the heads joined:
        ##                     (N, T, 4096) -> o_proj -> (N, T, 2560); + the layer's input (residual connection)
        ##     feed-forward:   RMSNorm; down_proj(SiLU(gate_proj(x)) * up_proj(x)): 2560 -> 9728 -> 2560; + residual
        ##   a final RMSNorm -> last_hidden_state (N, T, 2560)
        ## Every q_proj ... down_proj above is now a LoRALinear. use_cache=False: the cache keeps each layer's keys and
        ## values for generating text one token at a time; we read each post once, so it would only waste memory.
        hidden = self.qwen(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state
        last = attention_mask.sum(dim=1) - 1   ## (N,): each post's number of real tokens - 1 = where its <|endoftext|> is
        ## Indexing with two (N,) tensors picks hidden[0, last[0]], hidden[1, last[1]], ...: (N, T, 2560) -> (N, 2560).
        ## torch.arange(N) is the tensor [0, 1, ..., N-1], created on the same device as hidden.
        vec = hidden[torch.arange(len(last), device=hidden.device), last]   ## (N, 2560)
        return self.head(self.dropout(vec))    ## (N, 2560) -> dropout -> head -> (N, 20)

model = QwenClassifier(qwen, n_classes).to(device)   ## .to moves the new head into GPU memory (Qwen is there already)
## The tensors the optimizer will update: 252 A's, 252 B's, and the head's weight and bias
trainable = [p for p in model.parameters() if p.requires_grad]
n_all, n_trainable = sum(p.numel() for p in model.parameters()), sum(p.numel() for p in trainable)
print(f"{n_all / 1e6:,.0f}M parameters, {n_trainable / 1e6:.1f}M of them trainable ({n_trainable / n_all * 100:.2f}%), "
      f"in {len(trainable)} tensors")
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")   ## mostly the 8 GB of frozen weights

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 160.22it/s]

4,055M parameters, 33.1M of them trainable (0.82%), in 506 tensors
GPU memory in use: 8.2 GB


## Look inside a LoRA layer

First, a toy `LoRALinear` small enough to print ($W$ is 4 × 6, $r = 2$), to check that it computes what the equation says:
- At the start ($B = 0$), its output is exactly the frozen layer's.
- Once training has changed $B$, it computes $Wx + \frac{\alpha}{r}BAx$.
- $\frac{\alpha}{r}BA$ is a full 4 × 6 matrix, the same shape as $W$, but its rank is at most $r = 2$. After training it can be added into $W$ once ($W' = W + \frac{\alpha}{r}BA$), and the layer becomes a plain `nn.Linear` again, as fast as before. That's J&M's point that LoRA doesn't add any time during inference. (We don't merge here.)

Then the real thing: the query projection of Qwen's first layer.

In [6]:
## A toy: W is (4, 6), r = 2, alpha = 4 (so alpha / r = 2), no dropout. On the CPU, all 32-bit.
toy = LoRALinear(nn.Linear(6, 4, bias=False), r=2, alpha=4, dropout=0.0)
x = torch.randn(6)                                     ## one input vector, (6,)
W, A, B = toy.base.weight, toy.A.weight, toy.B.weight  ## (4, 6), (2, 6), (4, 2)
print("W", tuple(W.shape), " A", tuple(A.shape), " B", tuple(B.shape))
print("B = 0, so the output is exactly the frozen layer's:", torch.equal(toy(x), toy.base(x)))

## Pretend that training has changed B: fill it with random numbers. Changing a trainable weight in place needs
## torch.no_grad(); otherwise PyTorch stops with "a leaf Variable that requires grad is being used in an in-place
## operation", since autograd can't track such a change.
with torch.no_grad():
    B.normal_()        ## in place (trailing _): every entry of B drawn from a standard normal distribution
    ## @ is matrix multiplication; W @ x is (4,)
    print("now it computes W x + (alpha/r) B A x:", torch.allclose(toy(x), W @ x + toy.scale * (B @ (A @ x))))
    delta_W = toy.scale * (B @ A)                       ## (4, 2) @ (2, 6) -> (4, 6): the same shape as W
    print("(alpha/r) B A is", tuple(delta_W.shape), "with rank", torch.linalg.matrix_rank(delta_W).item())
    print("merged into one matrix, (W + (alpha/r) B A) x gives the same output:", torch.allclose((W + delta_W) @ x, toy(x)))

## The real thing: layer 1's query projection, now a LoRALinear
q = model.qwen.layers[0].self_attn.q_proj
for name, w in [("W", q.base.weight), ("A", q.A.weight), ("B", q.B.weight)]:
    print(f"{name}: shape {str(tuple(w.shape)):13s} {str(w.dtype):15s} trainable: {w.requires_grad}")
n_W, n_AB = q.base.weight.numel(), q.A.weight.numel() + q.B.weight.numel()
print(f"A and B together: {n_AB:,} numbers, {n_AB / n_W * 100:.1f}% of W's {n_W:,}")
x = torch.randn(3, q.base.in_features, device=device)   ## 3 random input vectors, (3, 2560)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    print("LoRA's part of the output, (alpha/r) B A x, is all zeros:", (q.B(q.A(x)) * q.scale == 0).all().item())

W (4, 6)  A (2, 6)  B (4, 2)
B = 0, so the output is exactly the frozen layer's: True
now it computes W x + (alpha/r) B A x: True
(alpha/r) B A is (4, 6) with rank 2
merged into one matrix, (W + (alpha/r) B A) x gives the same output: True
W: shape (4096, 2560)  torch.bfloat16  trainable: False
A: shape (16, 2560)    torch.float32   trainable: True
B: shape (4096, 16)    torch.float32   trainable: True
A and B together: 106,496 numbers, 1.0% of W's 10,485,760


LoRA's part of the output, (alpha/r) B A x, is all zeros: True


## Optimizer and learning-rate schedule

The BERT paper's recipe, as in the first RoBERTa runs: AdamW with weight decay 0.01, warmup over the first 10% of the steps, then linear decay to 0, and 3 epochs. Two changes:
- **The optimizer gets only the trainable tensors** ($A$, $B$ and the head). AdamW's two running averages per weight cost 2 × 33M numbers (0.26 GB) instead of 2 × 4B (32 GB).
- **Learning rate 1e-4** instead of 2e-5 (see "Compared with fine-tuning RoBERTa").

In [7]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 3
optimizer = torch.optim.AdamW(trainable, lr=1e-4, weight_decay=0.01)   ## only A, B and the head
steps_per_epoch = math.ceil(len(ids_tr) / BATCH_SIZE)                   ## 625 (one step = one batch)
total_steps = EPOCHS * steps_per_epoch                                   ## 1875
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()
print(f"{steps_per_epoch} steps per epoch, {total_steps} in total, {total_steps // 10} of them warmup")

625 steps per epoch, 1875 in total, 187 of them warmup


## One step in slow motion

Four training posts go through the forward pass, the loss and the backward pass, printing every shape, in training mode as in the real loop. No weights change: there's no `optimizer.step()`, and the gradients are thrown away at the end.

**Predict before running:** after `loss.backward()`, which of these will have no gradient at all, which a gradient that is all zeros, and which a nonzero one?
- $W$ in layer 1's q_proj
- the $A$'s
- the $B$'s
- the head's weight

Hint: right now $B = 0$. Write out $\frac{\partial \text{loss}}{\partial A}$ for $y = Wx + \frac{\alpha}{r}BAx$.

In [8]:
demo = np.arange(4)                                          ## the first 4 posts of the training part
input_ids, attention_mask = pad([ids_tr[i] for i in demo])   ## both (N, T), with N = 4
labels = torch.tensor(y_tr[demo], device=device)             ## (N,)
print("input_ids      ", tuple(input_ids.shape), input_ids.dtype)
print("attention_mask ", tuple(attention_mask.shape), attention_mask.dtype)

model.train()   ## as in training: dropout on, gradient checkpointing on
## The forward pass, one piece at a time (model(input_ids, attention_mask) does all four lines at once)
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    hidden = model.qwen(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state
    last = attention_mask.sum(dim=1) - 1
    vec = hidden[torch.arange(len(last), device=device), last]
    logits = model.head(model.dropout(vec))
print("hidden         ", tuple(hidden.shape), hidden.dtype)   ## (N, T, 2560), bfloat16 like Qwen's weights
print("last           ", last.tolist(), " (each post's number of real tokens - 1)")
## The token at each of those positions, picked the same way as vec: it should be <|endoftext|> every time
print("tokens there   ", tokenizer.convert_ids_to_tokens(input_ids[torch.arange(len(last), device=device), last].tolist()))
print("vec            ", tuple(vec.shape), vec.dtype)         ## (N, 2560)
print("logits         ", tuple(logits.shape), logits.dtype)   ## (N, 20)

## With a random head the loss starts well above RoBERTa's -log(1/20) = 3.0: about 4 averaged over all the training
## posts, and it varies a lot over just 4 posts. Qwen's vectors are long (norm about 140), so the random head's 20
## scores aren't all close to 0, and some wrong classes get high probabilities.
loss = criterion(logits.float(), labels)
print("loss           ", tuple(loss.shape), round(loss.item(), 3))

## The backward pass. With checkpointing, each layer's forward pass is rerun here, just before backpropagating through it.
loss.backward()

q = model.qwen.layers[0].self_attn.q_proj
print("\nW in layer 1's q_proj: .grad is", q.base.weight.grad)
## model.modules() visits every submodule, nested ones too; isinstance keeps the LoRALinear ones (252)
lora_layers = [m for m in model.modules() if isinstance(m, LoRALinear)]
for name in ["A", "B"]:
    ## getattr(m, name) is m.A or m.B. (grad != 0).any() is True if any entry isn't 0; sum() counts the Trues.
    n = sum((getattr(m, name).weight.grad != 0).any().item() for m in lora_layers)
    print(f"{name}: nonzero gradient in {n} of {len(lora_layers)} LoRA layers (in q_proj, its .grad is "
          f"{tuple(getattr(q, name).weight.grad.shape)})")
print("head weight: .grad", tuple(model.head.weight.grad.shape), " nonzero:", (model.head.weight.grad != 0).any().item())

## Why: LoRA adds (alpha/r) B h to the output, with h = A x (r numbers per token). By the chain rule,
##   d(loss)/dh = (alpha/r) Bᵀ d(loss)/dy = 0, because B = 0, so d(loss)/dA = d(loss)/dh xᵀ = 0 too.
##   B's gradient, (alpha/r) d(loss)/dy hᵀ, isn't zero: A is random, so h = A x isn't zero.
## After the first optimizer step B isn't zero anymore, and the A's start getting gradients too.
## W has no gradient at all: it's frozen (requires_grad=False), so autograd never computes one.
model.zero_grad(set_to_none=True)   ## throw the gradients away: this was only a demo

input_ids       (4, 461) torch.int64
attention_mask  (4, 461) torch.int64


hidden          (4, 461, 2560) torch.bfloat16
last            [147, 141, 460, 50]  (each post's number of real tokens - 1)
tokens there    ['<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>']
vec             (4, 2560) torch.bfloat16
logits          (4, 20) torch.bfloat16
loss            () 5.28



W in layer 1's q_proj: .grad is None
A: nonzero gradient in 0 of 252 LoRA layers (in q_proj, its .grad is (16, 2560))
B: nonzero gradient in 252 of 252 LoRA layers (in q_proj, its .grad is (4096, 16))
head weight: .grad (20, 2560)  nonzero: True


## Evaluation

As in the RoBERTa notebook: `model.eval()` turns dropout off (and gradient checkpointing, since there's no backward pass), and `torch.no_grad()` keeps no intermediate results. The posts go through sorted by length, 16 at a time, so each batch holds posts of similar length, split into the same pieces as in training. The predictions are put back in the original order.

**Why pieces, with no backward pass?** The first run of this notebook ran out of GPU memory in the dev evaluation after epoch 1, when evaluation used whole batches of 16 posts (up to 8,192 tokens):
- PyTorch keeps GPU memory it has used as a cache of blocks, and reuses them for new tensors that fit.
- Tensors four times bigger than any in training didn't fit the cached blocks, so PyTorch reserved new ones. Measured: the GPU then held 13.4 GB, against 11.3 GB during training.
- In the training pieces, evaluation fits in the blocks training already reserved.

In [9]:
## Returns (mean loss, accuracy, predicted classes) for a list of token-id lists and their labels
def evaluate(seqs, labels):
    model.eval()
    lengths = np.array([len(s) for s in seqs])     ## each post's length in tokens
    order = np.argsort(lengths)                    ## the posts' positions, shortest post first
    logits = torch.empty((len(seqs), n_classes))   ## (number of posts, 20), in CPU memory, filled in below
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for i in range(0, len(order), 16):                    ## 16 posts of similar length at a time,
            for idx in pieces(order[i:i + 16], lengths):      ## in the training pieces: idx = where they are in seqs
                ## pad(...) returns a pair; the * passes its two parts as the two arguments, model(input_ids, attention_mask).
                ## logits[idx] = ... writes rows idx[0], idx[1], ...: each post's scores go back to its original position.
                logits[idx] = model(*pad([seqs[j] for j in idx])).float().cpu()
    labels = torch.tensor(labels)                  ## (number of posts,)
    pred = logits.argmax(dim=1)                    ## each post's predicted class
    return criterion(logits, labels).item(), (pred == labels).float().mean().item(), pred.numpy()

## Training loop

The RoBERTa notebook's steps (zero the gradients, forward, loss, backward, clip, step, schedule), with gradient accumulation:
- `optimizer.zero_grad()` once per batch, then the forward and backward pass for each piece, then clipping and one `optimizer.step()`.
- Each piece's loss is its mean cross-entropy times its share of the batch (its posts / the batch's posts). Summed over the pieces, that's the batch's mean loss, so the summed gradients are the gradient of the batch's mean loss.

**Which epoch's weights: the last one's,** decided before running. The dev set is scored after each epoch (about half a minute), but only to watch the training:
- In the RoBERTa sweep (`fine-tuning-results.md`), dev loss bottomed out at epoch 2 or 3 and then rose while dev accuracy kept improving, so "lowest dev loss" picks too early.
- Picking the epoch with the best of several noisy dev accuracies can land on a bad epoch too.
- The BERT paper also trains a fixed 3 epochs and uses the final weights.

At the end, the trainable weights are saved to `qwen3-embedding-4b-lora.pt`. That's only 33M numbers (132 MB), since the 4 billion frozen ones never change; the whole model would be 8 GB. Together with the original Qwen in the Hugging Face cache, that file is the fine-tuned model, so it can be used again without retraining. That's what J&M means by LoRA modules that can be swapped in and out.

Time: 16.4 minutes per epoch on the 5070 Ti.

In [10]:
import time

start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()        ## dropout on, gradient checkpointing on
    train_loss = 0.0
    for step, batch in enumerate(train_batches(), start=1):   ## batch: (16,) positions in ids_tr (3 in the last one)
        optimizer.zero_grad()    ## clear .grad once per batch, not once per piece: the pieces' gradients must add up

        batch_loss = 0.0
        for piece in pieces(batch, lengths_tr):    ## 1 to 4 pieces, each at most TOKENS_PER_PIECE tokens
            input_ids, attention_mask = pad([ids_tr[i] for i in piece])   ## both (N, T)
            labels = torch.tensor(y_tr[piece], device=device)             ## (N,)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(input_ids, attention_mask)                 ## (N, 20)
            ## This piece's share of the batch's mean loss: the piece's mean times N / 16
            loss = criterion(logits.float(), labels) * len(piece) / len(batch)
            loss.backward()        ## .grad += this piece's gradient, for the A's, B's and the head
            batch_loss += loss.item()

        ## The batch is done: .grad now holds the gradient of its mean loss
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)   ## clip the gradient (all 33M numbers as one vector) to length 1.0
        optimizer.step()                                 ## update the A's, B's and the head
        scheduler.step()                                 ## move the learning rate one step along its schedule

        train_loss += batch_loss * len(batch)   ## the batch's mean loss times its posts = its total loss
        if step % 50 == 0:
            print(f"  epoch {epoch}, step {step}/{steps_per_epoch}: loss {batch_loss:.3f}, "
                  f"learning rate {scheduler.get_last_lr()[0]:.1e}  ({(time.perf_counter() - start) / 60:.1f} min)")

    ## Dev scores, only to watch the training: the weights we keep are the last epoch's, whatever these say
    dev_loss, dev_acc, _ = evaluate(ids_dev, y_dev)
    print(f"epoch {epoch}: train loss {train_loss / len(ids_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}  "
          f"({(time.perf_counter() - start) / 60:.1f} min so far)")

## Save only the trainable weights, copied to CPU memory: the frozen ones are Qwen's originals. named_parameters()
## yields (name, tensor) pairs, with names like "qwen.layers.0.self_attn.q_proj.A.weight".
## To use the file later: build the model as above, then model.load_state_dict(torch.load(...), strict=False).
## strict=False because the file has only the trainable weights, so the frozen ones are "missing" from it.
lora_state = {name: p.detach().cpu() for name, p in model.named_parameters() if p.requires_grad}
torch.save(lora_state, "qwen3-embedding-4b-lora.pt")   ## 132 MB: the A's, B's and the head
print(f"Training took {(time.perf_counter() - start) / 60:.0f} min. "
      f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

  epoch 1, step 50/625: loss 2.239, learning rate 2.7e-05  (1.1 min)


  epoch 1, step 100/625: loss 1.218, learning rate 5.3e-05  (2.4 min)


  epoch 1, step 150/625: loss 0.619, learning rate 8.0e-05  (3.8 min)


  epoch 1, step 200/625: loss 0.827, learning rate 9.9e-05  (5.0 min)


  epoch 1, step 250/625: loss 0.752, learning rate 9.6e-05  (6.5 min)


  epoch 1, step 300/625: loss 0.219, learning rate 9.3e-05  (7.7 min)


... (30 outputs left out)

  epoch 3, step 550/625: loss 0.031, learning rate 4.4e-06  (46.9 min)


  epoch 3, step 600/625: loss 0.010, learning rate 1.5e-06  (48.2 min)


epoch 3: train loss 0.146  dev loss 0.897  dev acc 82.24  (49.4 min so far)


Training took 49 min. Peak GPU memory: 9.6 GB


## Test set, once

In [11]:
from sklearn import metrics

test_loss, test_acc, test_pred = evaluate(ids_test, y_test)   ## 7,370 posts: about 3 minutes
print(f"LoRA-fine-tuned Qwen3-Embedding-4B test accuracy: {test_acc*100:.2f}  "
      f"(macro-F1 {metrics.f1_score(y_test, test_pred, average='macro')*100:.2f})\n")

rows = [("RoBERTa-base <s>, frozen + MLP (tuned, mean of 10 seeds)", 68.33),
        ("RoBERTa-base, all weights fine-tuned (RoBERTa recipe, mean of 4 runs)", 72.69),
        ("Sada et al.: Qwen3-8B chat model, 4-bit, LoRA on attention, 256 tokens", 77.75),
        ("Qwen3-Embedding-4B, frozen + LR (tuned)", 78.00),
        ("Qwen3-Embedding-4B, frozen + MLP (tuned, 10-seed ensemble)", 78.97),
        ("Qwen3-Embedding-4B, LoRA fine-tuned (this notebook, one run)", test_acc * 100)]
for name, acc in rows:
    print(f"{name:72s} {acc:6.2f}")
print(f"(1 point = {len(y_test) / 100:.0f} test messages. The frozen Qwen rows read 2,048 tokens per post, this one 512.)")

LoRA-fine-tuned Qwen3-Embedding-4B test accuracy: 78.68  (macro-F1 77.58)

RoBERTa-base <s>, frozen + MLP (tuned, mean of 10 seeds)                  68.33
RoBERTa-base, all weights fine-tuned (RoBERTa recipe, mean of 4 runs)     72.69
Sada et al.: Qwen3-8B chat model, 4-bit, LoRA on attention, 256 tokens    77.75
Qwen3-Embedding-4B, frozen + LR (tuned)                                   78.00
Qwen3-Embedding-4B, frozen + MLP (tuned, 10-seed ensemble)                78.97
Qwen3-Embedding-4B, LoRA fine-tuned (this notebook, one run)              78.68
(1 point = 74 test messages. The frozen Qwen rows read 2,048 tokens per post, this one 512.)


**Result:** 78.68 on test (macro-F1 77.58), one run.
- **About the same as the best frozen result.** It's 0.29 below the frozen 4B + tuned MLP ensemble (78.97) and 0.68 above the frozen 4B + tuned LR (78.00). One point is 74 test posts, and the 4 RoBERTa-recipe fine-tuning runs had a standard deviation of 0.92, so one LoRA run can't tell these apart. (The frozen rows also differ in other ways: they read 2,048 tokens, and their classifiers were tuned by CV.)
- **Fine-tuning helped Qwen far less than RoBERTa.** RoBERTa gained about 5 points over LR on its frozen vectors (67.23 → 72.69); Qwen gained under 1 (78.00 → 78.68). A likely reason: Qwen3-Embedding was already trained to put a summary of the text into its last token's vector, and the instruction already points it at this task, while RoBERTa's `<s>` was never trained to summarize anything.
- **0.93 above Sada et al.'s LoRA fine-tune of Qwen3-8B (77.75),** with half the parameters. Theirs is the general chat model, in 4-bit, with LoRA on the attention matrices only and 256 tokens. That's one run each, not tested for significance.
- **Dev loss and dev accuracy disagreed again, as in the RoBERTa sweep.** Dev loss was lowest after epoch 1 (0.631) and then rose to 0.897, while dev accuracy rose from 80.70 to 82.24. The train loss fell to 0.146: the model grew very confident on the training posts, so each dev mistake costs more loss. The old "lowest dev loss" rule would have kept epoch 1, 1.5 dev points lower; this notebook kept the last epoch.
- **Dev (82.24) is 3.6 points above test,** as in every notebook so far: dev comes from the same period as the training posts, test from later.
- **Cost:** 49 minutes of training (vs. about 4 for RoBERTa-base) and 9.6 GB of GPU memory at most.

The trained A's, B's and head are in `qwen3-embedding-4b-lora.pt`. A full fine-tune of the same model is set up in `Fine-tuned Qwen3-Embedding-4B Colab.ipynb` (it needs an 80 GB GPU). All fine-tuning runs: `fine-tuning-results.md`.